In [30]:
xml_file = "/mnt/data/Base Model_Arch vs Structural Clashes.xml"

In [29]:
from pathlib import Path
print(Path.cwd())
print(list(Path(".").rglob("*.xml")))

/home/b6b335a4-804b-48f8-a734-520d49ff1e55
[PosixPath('Base_Model_Arch_vs_Structural_Clashes.xml'), PosixPath('.ipynb_checkpoints/Base_Model_Arch_vs_Structural_Clashes-checkpoint.xml')]


In [33]:
import json
import xml.etree.ElementTree as ET
from pathlib import Path

xml_file = "Base_Model_Arch_vs_Structural_Clashes.xml"
output_json = "clashes_normalized.json"


def text_or_none(node, path):
    child = node.find(path)
    return child.text.strip() if child is not None and child.text else None


def to_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def parse_name_value_nodes(parent, path):
    out = {}
    for item in parent.findall(path):
        name = text_or_none(item, "name")
        value = text_or_none(item, "value")
        if name:
            out[name] = value
    return out


def extract_created_at(clashresult):
    date_node = clashresult.find("./createddate/date")
    if date_node is None:
        return None
    try:
        return (
            f"{int(date_node.attrib.get('year')):04d}-"
            f"{int(date_node.attrib.get('month')):02d}-"
            f"{int(date_node.attrib.get('day')):02d}T"
            f"{int(date_node.attrib.get('hour', 0)):02d}:"
            f"{int(date_node.attrib.get('minute', 0)):02d}:"
            f"{int(date_node.attrib.get('second', 0)):02d}"
        )
    except Exception:
        return None


def extract_clash_point(clashresult):
    pt = clashresult.find("./clashpoint/pos3f")
    if pt is None:
        return None
    return {
        "x": to_float(pt.attrib.get("x")),
        "y": to_float(pt.attrib.get("y")),
        "z": to_float(pt.attrib.get("z")),
    }


def extract_revit_global_id(attrs):
    candidate_keys = [
        "Revit UniqueId",
        "Revit Unique ID",
        "UniqueId",
        "Unique ID",
        "GlobalId",
        "Global ID",
        "IfcGUID",
        "Ifc GUID",
    ]
    for key in candidate_keys:
        if key in attrs and attrs[key]:
            return attrs[key]
    return None


def parse_clash_object(clashobject):
    attrs = parse_name_value_nodes(clashobject, "./objectattribute")
    tags = parse_name_value_nodes(clashobject, "./smarttags/smarttag")
    return {
        "revitGlobalId": extract_revit_global_id(attrs),
        "elementId": attrs.get("Element ID"),
        "clashMetadata": {
            "layer": text_or_none(clashobject, "./layer"),
            "itemName": tags.get("Item Name"),
            "itemType": tags.get("Item Type"),
        },
        "rawAttributes": attrs,
        "rawSmartTags": tags,
    }


def parse_clash_result(clashresult):
    return {
        "clashName": clashresult.attrib.get("name"),
        "clashGuid": clashresult.attrib.get("guid"),
        "clashMetadata": {
            "status": clashresult.attrib.get("status"),
            "description": text_or_none(clashresult, "./description"),
            "resultStatus": text_or_none(clashresult, "./resultstatus"),
            "distance": to_float(clashresult.attrib.get("distance")),
            "href": clashresult.attrib.get("href"),
            "gridLocation": text_or_none(clashresult, "./gridlocation"),
            "createdAt": extract_created_at(clashresult),
            "clashPoint": extract_clash_point(clashresult),
        },
        "objects": [
            parse_clash_object(obj)
            for obj in clashresult.findall("./clashobjects/clashobject")
        ],
    }


def parse_clash_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    output = {
        "sourceFile": Path(xml_path).name,
        "sourcePath": str(Path(xml_path).resolve()),
        "units": root.attrib.get("units"),
        "tests": []
    }

    for clashtest in root.findall(".//clashtest"):
        summary = clashtest.find("./summary")

        test_data = {
            "testName": clashtest.attrib.get("name"),
            "testType": clashtest.attrib.get("test_type"),
            "testStatus": clashtest.attrib.get("status"),
            "tolerance": clashtest.attrib.get("tolerance"),
            "summary": {
                "total": int(summary.attrib["total"]) if summary is not None and summary.attrib.get("total") else None,
                "new": int(summary.attrib["new"]) if summary is not None and summary.attrib.get("new") else None,
                "active": int(summary.attrib["active"]) if summary is not None and summary.attrib.get("active") else None,
                "reviewed": int(summary.attrib["reviewed"]) if summary is not None and summary.attrib.get("reviewed") else None,
                "approved": int(summary.attrib["approved"]) if summary is not None and summary.attrib.get("approved") else None,
                "resolved": int(summary.attrib["resolved"]) if summary is not None and summary.attrib.get("resolved") else None,
            },
            "clashes": []
        }

        for clashresult in clashtest.findall("./clashresults/clashresult"):
            test_data["clashes"].append(parse_clash_result(clashresult))

        output["tests"].append(test_data)

    return output


def build_usable_clash_objects(parsed_data):
    usable = []

    for test in parsed_data.get("tests", []):
        for clash in test.get("clashes", []):
            objects = clash.get("objects", [])

            usable.append({
                "testName": test.get("testName"),
                "testType": test.get("testType"),
                "testStatus": test.get("testStatus"),
                "clashName": clash.get("clashName"),
                "clashGuid": clash.get("clashGuid"),
                "clashMetadata": clash.get("clashMetadata"),
                "objects": objects,
                "objectA": objects[0] if len(objects) > 0 else None,
                "objectB": objects[1] if len(objects) > 1 else None,
            })

    return usable


data = parse_clash_xml(xml_file)

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

usable_clashes = build_usable_clash_objects(data)

with open("usable_clashes.json", "w", encoding="utf-8") as f:
    json.dump(usable_clashes, f, indent=2, ensure_ascii=False)

print("Using:", Path(xml_file).resolve())
print("Done. Wrote:", Path(output_json).resolve())
print("Done. Wrote:", Path("usable_clashes.json").resolve())
print(f"Total usable clash objects: {len(usable_clashes)}")

print("\nFirst 3 usable clash objects:")
print(json.dumps(usable_clashes[:3], indent=2, ensure_ascii=False))

Using: /home/b6b335a4-804b-48f8-a734-520d49ff1e55/Base_Model_Arch_vs_Structural_Clashes.xml
Done. Wrote: /home/b6b335a4-804b-48f8-a734-520d49ff1e55/clashes_normalized.json
Done. Wrote: /home/b6b335a4-804b-48f8-a734-520d49ff1e55/usable_clashes.json
Total usable clash objects: 3987

First 3 usable clash objects:
[
  {
    "testName": "Arch vs Struct",
    "testType": "hard",
    "testStatus": "ok",
    "clashName": "Clash1",
    "clashGuid": "f99fba96-1ab8-4f60-bc31-407511924e0d",
    "clashMetadata": {
      "status": "active",
      "description": "Hard",
      "resultStatus": "Active",
      "distance": -4.216,
      "href": "Base Model_Arch vs Structural Clashes_files\\cd000001.jpg",
      "gridLocation": "B-6 : L3",
      "createdAt": "2026-04-03T09:28:56",
      "clashPoint": {
        "x": 417625.629,
        "y": 78709.244,
        "z": 247.548
      }
    },
    "objects": [
      {
        "revitGlobalId": null,
        "elementId": "748873",
        "clashMetadata": {
        